In [4]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2, ResNet50, EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import os

In [5]:
# --- 1. Data Preparation & Augmentation (Kaggle Style) ---
TRAIN_DIR = 'dataset/train'
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

In [6]:
# Data Augmentation helps reach the 98% accuracy mentioned in your source
train_datagen = ImageDataGenerator(
    rescale=1./255.,
    rotation_range=15,
    shear_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2 # Use 20% for validation
)

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training'
)

validation_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation'
)

Found 6400 images belonging to 2 classes.
Found 1600 images belonging to 2 classes.


In [7]:
# --- 2. Model Builder Function ---
def build_transfer_model(base_model_class, name):
    print(f"Building {name}...")
    
    # Load pre-trained base (ImageNet), exclude top
    base = base_model_class(weights='imagenet', include_top=False, input_shape=(*IMG_SIZE, 3))
    
    # Freeze the base layers
    base.trainable = False
    
    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.5)(x) # Prevents overfitting
    predictions = Dense(1, activation='sigmoid')(x) # Binary output
    
    model = Model(inputs=base.input, outputs=predictions)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [8]:
# --- 3. Comparison Loop ---
architectures = [
    (MobileNetV2, 'MobileNetV2'),
    (ResNet50, 'ResNet50'),
    (EfficientNetB0, 'EfficientNetB0')
]

# Callbacks for high performance (Early stopping)
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=2)
]

results = []

for arch, name in architectures:
    model = build_transfer_model(arch, name)
    
    # Train
    history = model.fit(
        train_generator,
        epochs=10,
        validation_data=validation_generator,
        callbacks=callbacks,
        verbose=1
    )
    
    # Record Metrics
    acc = history.history['accuracy'][-1]
    val_acc = history.history['val_accuracy'][-1]
    
    results.append({'Model': name, 'Train Acc': acc, 'Val Acc': val_acc})
    
    # Save the best model for Streamlit
    if name == 'EfficientNetB0':
        model.save('best_model.keras')
        print("Saved EfficientNetB0 for deployment.")

print("\n--- Comparison Results ---")
print(results)

Building MobileNetV2...
Epoch 1/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 255s 1s/step - accuracy: 0.9588 - loss: 0.1131 - val_accuracy: 0.9787 - val_loss: 0.0578 - learning_rate: 0.0010
Epoch 2/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 278s 1s/step - accuracy: 0.9689 - loss: 0.0848 - val_accuracy: 0.9700 - val_loss: 0.0811 - learning_rate: 0.0010
Epoch 3/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 208s 1s/step - accuracy: 0.9722 - loss: 0.0740 - val_accuracy: 0.9781 - val_loss: 0.0572 - learning_rate: 0.0010
Epoch 4/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 279s 1s/step - accuracy: 0.9798 - loss: 0.0508 - val_accuracy: 0.9775 - val_loss: 0.0513 - learning_rate: 5.0000e-04
Building ResNet50...
Epoch 1/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 625s 3s/step - accuracy: 0.5469 - loss: 0.7089 - val_accuracy: 0.6056 - val_loss: 0.6634 - learning_rate: 0.0010
Epoch 2/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 571s 3s/step - accuracy: 0.5802 - loss: 0.6729 - val_accuracy: 0.6125 - val_loss: 0.6613 - learning_rate: 0.0010
Epoch 3/10
200/200 ━━━━━━━━━━━━━━